In [4]:
import pandas as pd
import numpy as np
import json
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

df = pd.read_csv('model_table.csv')
df['visit_date'] = pd.to_datetime(df['visit_date'])


# Predict claim outcome (Paid, Pending, Rejected) BEFORE submission.
# Exclude payment_days and approved_amount (Leakage).

from sklearn.preprocessing import LabelEncoder

# Encode the target variable
le = LabelEncoder()
y = pd.Series(le.fit_transform(df['claim_status']))

target_mapping = dict(zip(le.transform(le.classes_), le.classes_))
print("Target Mapping:", target_mapping)
target_names = [target_mapping[i] for i in range(len(target_mapping))]

# Features for predicting Claim Outcome (BEFORE submission)
features = [
    'billed_amount', 'provider_rejection_rate', 'high_billed_flag',
    'department_enc', 'insurance_provider_enc', 'visit_type_enc',
    'age', 'length_of_stay_hours', 'chronic_flag'
]

X = df[['visit_date'] + features].copy()

X.fillna(X.median(numeric_only=True), inplace=True)


X = X.sort_values('visit_date')
y = y.loc[X.index]

split_idx = int(len(X) * 0.8)
X_train = X.iloc[:split_idx].drop(columns=['visit_date'])
X_test  = X.iloc[split_idx:].drop(columns=['visit_date'])
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")


# 'Rejected' is the minority class (~15%), highly critical to predict.
print("Original y_train:\n", y_train.value_counts())
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("\nResampled y_train:\n", y_train_resampled.value_counts())

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. Scale the data so large numbers (billed_amount) don't break the math
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

# 2. Train the baseline model on the scaled data
baseline_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
baseline_model.fit(X_train_scaled, y_train_resampled)

# 3. Predict and evaluate
y_pred_base = baseline_model.predict(X_test_scaled)

print("=== Baseline Model (Logistic Regression) ===")
print(classification_report(y_test, y_pred_base, target_names=target_names))



rf_model = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42)
rf_model.fit(X_train_resampled, y_train_resampled)
y_pred_rf = rf_model.predict(X_test)

print("=== Advanced Model (Random Forest) ===")
print(classification_report(y_test, y_pred_rf, target_names=target_names))


joblib.dump(rf_model, 'claim_model.pkl')

schema = {
    "model_name": "Claim Outcome Classification",
    "features": list(X_train.columns),
    "target_mapping": {0: "Paid", 1: "Pending", 2: "Rejected"} # Verify encoding
}
with open('claim_feature_schema.json', 'w') as f:
    json.dump(schema, f)

print("\n claim_model.pkl and claim_feature_schema.json saved.")


Target Mapping: {0: 'Paid', 1: 'Pending', 2: 'Rejected'}
Train size: 20000 | Test size: 5000
Original y_train:
 0    11943
1     4988
2     3069
Name: count, dtype: int64

Resampled y_train:
 2    11943
1    11943
0    11943
Name: count, dtype: int64
=== Baseline Model (Logistic Regression) ===
              precision    recall  f1-score   support

        Paid       0.60      0.62      0.61      2997
     Pending       0.24      0.12      0.17      1275
    Rejected       0.19      0.32      0.23       728

    accuracy                           0.45      5000
   macro avg       0.34      0.36      0.34      5000
weighted avg       0.45      0.45      0.44      5000



C:\Users\ADMIN\anaconda3\envs\tf_gpu_env\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


=== Advanced Model (Random Forest) ===
              precision    recall  f1-score   support

        Paid       0.61      0.82      0.70      2997
     Pending       0.31      0.07      0.12      1275
    Rejected       0.21      0.21      0.21       728

    accuracy                           0.54      5000
   macro avg       0.38      0.37      0.34      5000
weighted avg       0.48      0.54      0.48      5000


 claim_model.pkl and claim_feature_schema.json saved.


## Why are we doing this?

### Business Purpose:
The goal of this model is to predict whether an insurance claim will be Paid, Pending, or Rejected (claim_status) prior to its submission to the payer. By flagging claims with a high probability of rejection, the finance team can review, recode, or add missing documentation, thereby directly reducing revenue leakage.

1. Feature Selection & Leakage Prevention
To accurately simulate the pre-submission environment, post-adjudication variables (approved_amount and payment_days) were strictly excluded. If included, they would cause target leakage. The model relies instead on:

    Financial & Insurer Context: billed_amount, high_billed_flag (top decile), and the historical provider_rejection_rate (which captures the strictness of the specific insurer).

    Clinical Complexity: length_of_stay_hours, department, and chronic conditions (which often trigger higher insurer scrutiny).

2. Time-Based Validation
Similar to Model A, a time-based 80/20 train/test split was used. This is especially critical in financial modeling, as payer behaviors, coding rules, and rejection rates evolve over time.

3. Class Imbalance Strategy
The "Rejected" class — the most critical class from a business revenue perspective — represented only ~15% of the data. To ensure the model does not simply bias toward the majority "Paid" class, SMOTE was applied to the training set to upsample the "Rejected" and "Pending" instances.

4. Model Training

    Baseline: A balanced Logistic Regression model was deployed to set a performance floor.

    Advanced: A Random Forest Classifier was trained to capture complex thresholds (e.g., insurers rejecting claims only when billed amounts exceed certain limits for specific departments). The resulting artifact (claim_model.pkl) and schema were saved for deployment in the hospital's revenue cycle management workflow.